# Demo LSTM Phân Tích Cảm Xúc Tiếng Việt
#### Nhóm 5:
- Cáp Kim Hải Anh - 23520036
- Hoàng Đức Dũng - 23520328
- Nguyễn Thái Sơn - 23521356
- Bùi Ngọc Thiên Thanh - 23521436

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## Tạo Dữ Liệu mãu
- `1`: tích cực
- `0`: tiêu cực

In [ ]:
positive_reviews = [
    "Sản phẩm rất tốt và chất lượng vượt mong đợi",
    "Mình cực kỳ hài lòng với dịch vụ của shop",
    "Giao hàng nhanh và đóng gói rất cẩn thận",
    "Nhân viên hỗ trợ nhiệt tình và thân thiện",
    "Máy chạy rất mượt và pin dùng lâu",
    "Đồ ăn ngon và phục vụ chuyên nghiệp",
    "Giá hợp lý so với chất lượng sản phẩm",
    "Ứng dụng dễ sử dụng và giao diện đẹp",
    "Tôi sẽ tiếp tục ủng hộ shop lần sau",
    "Sản phẩm giống mô tả và dùng rất ổn",
    "Khách sạn sạch sẽ và vị trí thuận tiện",
    "Chất lượng âm thanh rất tốt",
    "Mình rất thích thiết kế của sản phẩm này",
    "Dịch vụ chăm sóc khách hàng tuyệt vời",
    "Sản phẩm hoạt động ổn định và hiệu quả",
    "Trải nghiệm mua sắm rất hài lòng",
    "Quán cà phê có không gian đẹp và yên tĩnh",
    "Thời gian giao hàng nhanh hơn mong đợi",
    "Sản phẩm dùng rất bền và đáng tiền",
    "Nhân viên tư vấn rõ ràng và dễ hiểu",
    "Đóng gói kỹ càng và không bị hư hỏng",
    "Hiệu năng máy tính rất mạnh",
    "Mình cảm thấy rất hài lòng sau khi sử dụng",
    "Dịch vụ sửa chữa nhanh và chuyên nghiệp",
    "Phim rất hay và cảm động"
]

negative_reviews = [
    "Sản phẩm quá tệ và không giống mô tả",
    "Mình rất thất vọng về chất lượng",
    "Giao hàng chậm và đóng gói sơ sài",
    "Nhân viên phục vụ thiếu chuyên nghiệp",
    "Máy bị lỗi chỉ sau vài ngày sử dụng",
    "Đồ ăn nguội và không ngon",
    "Giá quá cao so với chất lượng",
    "Ứng dụng thường xuyên bị crash",
    "Tôi sẽ không mua lại sản phẩm này",
    "Sản phẩm bị hỏng khi vừa nhận hàng",
    "Khách sạn bẩn và phòng có mùi khó chịu",
    "Âm thanh rè và chất lượng kém",
    "Thiết kế sản phẩm xấu và bất tiện",
    "Dịch vụ chăm sóc khách hàng quá tệ",
    "Máy hoạt động chậm và hay đứng",
    "Trải nghiệm mua sắm rất khó chịu",
    "Quán đông và phục vụ quá lâu",
    "Thời gian giao hàng lâu hơn dự kiến",
    "Sản phẩm nhanh hỏng và không bền",
    "Nhân viên tư vấn không nhiệt tình",
    "Đóng gói cẩu thả làm sản phẩm bị móp",
    "Hiệu năng máy yếu và lag liên tục",
    "Mình không hài lòng với sản phẩm này",
    "Dịch vụ sửa chữa quá chậm",
    "Phim quá chán và nội dung nhạt nhẽo"
]

reviews = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

# Tạo DataFrame

df = pd.DataFrame({
    "review": reviews,
    "label": labels
})

print(df.head())

                                         review  label
0  Sản phẩm rất tốt và chất lượng vượt mong đợi      1
1     Mình cực kỳ hài lòng với dịch vụ của shop      1
2      Giao hàng nhanh và đóng gói rất cẩn thận      1
3     Nhân viên hỗ trợ nhiệt tình và thân thiện      1
4             Máy chạy rất mượt và pin dùng lâu      1


## Chia train / validation / test

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df["review"],
    df["label"],
    test_size=0.4,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

Train size: 30
Validation size: 10
Test size: 10


## Tokenization và Padding
1. Chuyển từ → số nguyên bằng `Tokenizer`
2. Padding để các câu có cùng độ dài

In [ ]:
MAX_WORDS = 5000
MAX_LEN = 20

# Khởi tạo tokenizer

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")

# Học vocabulary từ tập train

tokenizer.fit_on_texts(X_train)

# Chuyển text -> sequence

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print(X_train.iloc[0])
print(X_train_seq[0])

Sản phẩm giống mô tả và dùng rất ổn
[4, 5, 52, 53, 54, 2, 24, 3, 25]


In [ ]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print(X_train_pad.shape)
print(X_train_pad[0])

(30, 20)
[ 4  5 52 53 54  2 24  3 25  0  0  0  0  0  0  0  0  0  0  0]


## Xây dựng mô hình LSTM
Kiến trúc
* Embedding: biến token thành vector dense
* LSTM: học ngữ cảnh chuỗi
* Dense sigmoid: dự đoán nhị phân

In [ ]:
VOCAB_SIZE = len(tokenizer.word_index) + 1

model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    ),

    LSTM(64),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Huấn luyện

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=4,
    callbacks=[early_stop]
)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.4000 - loss: 0.6975 - val_accuracy: 0.6000 - val_loss: 0.6902
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3000 - loss: 0.6979 - val_accuracy: 0.4000 - val_loss: 0.6961
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3667 - loss: 0.6991 - val_accuracy: 0.6000 - val_loss: 0.6888
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5000 - loss: 0.6962 - val_accuracy: 0.6000 - val_loss: 0.6916
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5333 - loss: 0.6929 - val_accuracy: 0.6000 - val_loss: 0.6921
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6667 - loss: 0.6894 - val_accuracy: 0.6000 - val_loss: 0.6925


In [ ]:
# Predict probability

y_pred_prob = model.predict(X_test_pad)

# Convert probability -> label

y_pred = (y_pred_prob > 0.5).astype(int)

# Accuracy

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

# Classification report

print(classification_report(y_test, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
Accuracy: 0.3
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.30      1.00      0.46         3

    accuracy                           0.30        10
   macro avg       0.15      0.50      0.23        10
weighted avg       0.09      0.30      0.14        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Dự đoán câu mới

In [ ]:
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])

    pad = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding="post"
    )

    pred = model.predict(pad)[0][0]

    if pred >= 0.5:
        label = "Tích cực"
    else:
        label = "Tiêu cực"

    print(f"Câu: {text}")
    print(f"Xác suất tích cực: {pred:.4f}")
    print(f"Dự đoán: {label}")

In [ ]:
predict_sentiment("Sản phẩm dùng rất tốt và đáng tiền")

predict_sentiment("Dịch vụ quá tệ và tôi thất vọng")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
Câu: Sản phẩm dùng rất tốt và đáng tiền
Xác suất tích cực: 0.5114
Dự đoán: Tích cực
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Câu: Dịch vụ quá tệ và tôi thất vọng
Xác suất tích cực: 0.5113
Dự đoán: Tích cực
